# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**Lerner, Federico Ezequiel**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [95]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi
from aima_libs.aima import PriorityQueue # Agrego lib de colas prioritarias

In [96]:
def search_algorithm(number_disks=5) -> (NodeHanoi, dict):

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    # La heurística que voy a implementar se puede representar con la fórmula:
    # h = (discos que no se encuentran en el destino) + 2 * (discos incorrectos en el destino).
    # El "2 *" es porque cada disco incorrecto en el destino tiene un costo de 2 (sacarlo y volverlo a poner).
    # Esto busca representar que cada disco que no esté en la varilla de destino necesita al menos 1 movimiento,
    # y cada disco mal colocado en el destino necesita al menos 2 movimiento.

    # Función para evaluación de la cola de prioridad
    def f(nodo):
        estado = nodo.state
        varilla_destino = estado.rods[2]

        # Cuento cuántos discos están bien apilados desde el fondo del destino
        bien_colocados = 0
        while bien_colocados < len(varilla_destino) and varilla_destino[bien_colocados] == number_disks - bien_colocados:
            bien_colocados += 1

        # Cuento los discos que están en el destino pero mal colocados
        mal_colocados_en_destino = len(varilla_destino) - bien_colocados
        # Cuento los discos que ni siquiera llegaron al destino
        fuera_del_destino = number_disks - len(varilla_destino)

        h = fuera_del_destino + 2 * mal_colocados_en_destino
        return nodo.path_cost + h

    frontera = PriorityQueue(order='min', f=f)
    frontera.append(NodeHanoi(initial_state))
    explorados = set()

    nodos_explorados = 0
    profundidad_maxima = 0

    while len(frontera) != 0:
        prioridad, nodo = frontera.pop()
        nodos_explorados += 1

        # Si ya pasé por este estado antes, lo salteo para no repetirlo
        if nodo.state in explorados:
            continue
        explorados.add(nodo.state)

        if nodo.depth > profundidad_maxima:
            profundidad_maxima = nodo.depth

        # Si llegué al objetivo, devuelvo la solución junto con las métricas
        if problem.goal_test(nodo.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": nodos_explorados,
                "states_visited": len(explorados),
                "nodes_in_frontier": len(frontera),
                "max_depth": profundidad_maxima,
                "cost_total": nodo.state.accumulated_cost,
            }
            return nodo, metrics

        # Expando el nodo actual y agrego los hijos nuevos a la frontera
        for hijo in nodo.expand(problem):
            if hijo.state not in explorados:
                frontera.append(hijo)

    # Si no encontré solucion, devuelvo métricas
    metrics = {
        "solution_found": False,
        "nodes_explored": nodos_explorados,
        "states_visited": len(explorados),
        "nodes_in_frontier": len(frontera),
        "max_depth": profundidad_maxima,
        "cost_total": None,
    }
    
    return None, metrics
    

Se prueba la función:

In [97]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [98]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 223
states_visited: 158
nodes_in_frontier: 20
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [99]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [100]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
